# Quantum Wavepacket Simulation

This notebook simulates the time evolution of a quantum mechanical wavepacket under several potentials, and demonstrates **time-reversal symmetry** of the Schrödinger equation.

---

## 1. The Time-Dependent Schrödinger Equation

We solve the TDSE on a 1D spatial grid:

$$i\hbar \frac{\partial \psi}{\partial t} = \hat{H}\psi = \left[-\frac{\hbar^2}{2m}\frac{\partial^2}{\partial x^2} + V(x)\right]\psi$$

The wavefunction $\psi(x,t)$ is complex-valued. The physically observable quantity is the **probability density** $|\psi(x,t)|^2$, which integrates to 1 (normalisation).

---

## 2. Crank–Nicolson Discretisation

We discretise space with $N$ points separated by $\Delta x$ and time with step $\Delta t$. The second derivative is approximated by the standard centred finite difference:

$$\frac{\partial^2 \psi}{\partial x^2}\bigg|_j \approx \frac{\psi_{j+1} - 2\psi_j + \psi_{j-1}}{(\Delta x)^2}$$

The **Crank–Nicolson** method averages the Hamiltonian at time levels $n$ and $n+1$, giving a scheme that is unconditionally stable and **unitary** (norm-preserving):

$$\left(I + \frac{i\Delta t}{2\hbar}\hat{H}\right)\psi^{n+1} = \left(I - \frac{i\Delta t}{2\hbar}\hat{H}\right)\psi^{n}$$

Defining $\gamma = \frac{i\Delta t}{2\hbar}$ and $\kappa = \frac{\hbar^2}{2m(\Delta x)^2}$, the matrices $A$ and $B$ are **tridiagonal**:

$$A_{jj} = 1 + \gamma(2\kappa + V_j), \quad A_{j,j\pm1} = -\gamma\kappa$$
$$B_{jj} = 1 - \gamma(2\kappa + V_j), \quad B_{j,j\pm1} = +\gamma\kappa$$

Each timestep reduces to solving the tridiagonal linear system $A\,\psi^{n+1} = B\,\psi^{n}$.

---

## 3. The Thomas Algorithm

Because $A$ is tridiagonal, we solve $A\mathbf{x} = \mathbf{d}$ in $O(N)$ time using **Thomas's algorithm** (tridiagonal matrix algorithm), which is a specialised form of Gaussian elimination without pivoting.

Given lower diagonal $a$, main diagonal $b$, upper diagonal $c$, and RHS $d$:

**Forward sweep** (eliminate lower diagonal):
$$c'_1 = \frac{c_1}{b_1}, \quad d'_1 = \frac{d_1}{b_1}$$
$$c'_i = \frac{c_i}{b_i - a_i c'_{i-1}}, \quad d'_i = \frac{d_i - a_i d'_{i-1}}{b_i - a_i c'_{i-1}}$$

**Back substitution**:
$$x_N = d'_N, \quad x_i = d'_i - c'_i\, x_{i+1}$$

The tridiagonal structure of A means the diagonals can be precomputed once and reused every timestep. Only the RHS $\mathbf{d} = B\psi^n$ changes, so the forward-sweep modified diagonals $c'$ and the pivot denominators are cached.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FFMpegWriter

## 4. Grid and Physical Parameters

We work in natural units where $\hbar = m = 1$. The spatial domain is $x \in [-L/2,\, L/2]$ with $N$ uniformly-spaced points, giving $\Delta x = L/N$.

In [2]:
L = 120.0
N = 1200
dx = L / N
dt = 0.003
steps_per_frame = 10
hbar = 1.0
m = 1.0
x = np.linspace(-L / 2, L / 2, N)

## 5. Initial State: Gaussian Wavepacket

The initial wavefunction is a **Gaussian wavepacket** — a minimum-uncertainty state that simultaneously localises position and momentum as tightly as the Heisenberg uncertainty principle allows:

$$\psi(x, 0) = \mathcal{N}\exp\!\left(-\frac{(x-x_0)^2}{4\sigma^2}\right)\exp(ik_0 x)$$

where $x_0$ is the centre, $\sigma$ the spatial width, and $k_0$ the central wavenumber (so $p_0 = \hbar k_0$ is the mean momentum). The prefactor $\mathcal{N}$ normalises the state so $\int|\psi|^2\,dx = 1$.

In [3]:
def gaussian_wavepacket(x_vals, x0, sigma, k0):
    psi = np.exp(-(x_vals - x0) ** 2 / (4 * sigma ** 2)) * np.exp(1j * k0 * x_vals)
    norm = np.sqrt(np.sum(np.abs(psi) ** 2) * dx)
    return psi / norm

## 6. Solver: Thomas Algorithm for the Crank–Nicolson System

For the tridiagonal system $A\psi^{n+1} = \mathbf{d}$ with $\mathbf{d} = B\psi^n$, both $A$ and $B$ share the same off-diagonal magnitude $\gamma\kappa$ and differ only in the sign on their main diagonals.

Since the potential $V(x)$ is time-independent, the diagonals of $A$ never change. We therefore run the **forward sweep once** at construction time and cache the modified upper diagonal $c'$ and pivot denominators, reducing each subsequent timestep to a single back-substitution pass plus one $B\psi$ matrix-vector product.

The matrix $B$ is also tridiagonal, so $\mathbf{d} = B\psi$ is computed in $O(N)$ as:
$$d_j = B_{jj}\psi_j + B_{j,j-1}\psi_{j-1} + B_{j,j+1}\psi_{j+1}$$

In [4]:
def build_thomas_solver(V_vals):
    k_const = hbar ** 2 / (2 * m * dx ** 2)
    gamma = 1j * dt / (2 * hbar)

    main_A = 1.0 + gamma * (2 * k_const + V_vals)
    off_A  = -gamma * k_const * np.ones(N - 1)
    main_B = 1.0 - gamma * (2 * k_const + V_vals)
    off_B  =  gamma * k_const * np.ones(N - 1)

    # --- Thomas forward sweep on A (cached) ---
    c_prime = np.zeros(N - 1, dtype=complex)
    b_prime = main_A.copy().astype(complex)

    c_prime[0] = off_A[0] / b_prime[0]
    for i in range(1, N - 1):
        denom = b_prime[i] - off_A[i - 1] * c_prime[i - 1]
        c_prime[i] = off_A[i] / denom
        b_prime[i] = denom
    b_prime[N - 1] = main_A[N - 1] - off_A[N - 2] * c_prime[N - 2]

    def apply_B(psi):
        d = main_B * psi
        d[:-1] += off_B * psi[1:]
        d[1:]  += off_B * psi[:-1]
        return d

    def solve(psi):
        d = apply_B(psi).astype(complex)

        # Forward sweep on RHS
        d[0] /= main_A[0]
        for i in range(1, N):
            d[i] = (d[i] - off_A[i - 1] * d[i - 1]) / b_prime[i]

        # Back substitution
        for i in range(N - 2, -1, -1):
            d[i] -= c_prime[i] * d[i + 1]

        return d

    return solve

## 7. Potentials

We study four paradigmatic potentials:

| Scenario | $V(x)$ | Physics |
|---|---|---|
| Free particle | $0$ | Gaussian spreads via dispersion: $\sigma(t)\propto\sqrt{1+(t/2m\sigma^2)^2}$ |
| Harmonic oscillator | $\frac{1}{2}m\omega^2 x^2$ | Wavepacket oscillates at $\omega$ without spreading (coherent state) |
| Particle in a box | Hard walls at $|x|\gtrsim L/2$ | Reflection and interference; discrete energy levels |
| Potential barrier | Rectangular barrier of height $V_0$, width $w$ | **Quantum tunnelling**: transmission even for $E < V_0$ |

In [ ]:
free_V = np.zeros(N)

omega = 0.5
harmonic_V = 0.5 * m * omega ** 2 * x ** 2

wall_width = 30.0
wall_height = 300.0
box_V = np.zeros(N)
box_V[x < (-L / 2 + wall_width)] = wall_height
box_V[x > ( L / 2 - wall_width)] = wall_height

barrier_left  = 5.0
barrier_width = 10.0
barrier_right = barrier_left + barrier_width
barrier_height = 40
barrier_V = np.zeros(N)
barrier_V[(x > barrier_left) & (x < barrier_right)] = barrier_height

sim_configs = [
    {
        "title": "Potential Barrier",
        "V": barrier_V,
        "psi0": gaussian_wavepacket(x, -35.0, 3.0, 10),
        "markers": [
            {"type": "span",  "x0": barrier_left, "x1": barrier_right, "color": "#c9c9c9"},
            {"type": "vline", "x": barrier_left,  "color": "#666666", "label": "Barrier"},
            {"type": "vline", "x": barrier_right, "color": "#666666"},
        ],
    },
    {
        "title": "Harmonic Oscillator",
        "V": harmonic_V,
        "psi0": gaussian_wavepacket(x, -25.0, 1.0, 0.0),
        "markers": [
            {"type": "vline", "x": 0.0, "color": "#555555", "label": "Equilibrium"}
        ],
    },
    {
        "title": "Particle in a Box",
        "V": box_V,
        "psi0": gaussian_wavepacket(x, -20.0, 2.5, 7.0),
        "markers": [
            {"type": "span", "x0": -L / 2, "x1": -L / 2 + wall_width, "color": "#b5b5b5"},
            {"type": "span", "x0":  L / 2 - wall_width, "x1": L / 2,  "color": "#b5b5b5"},
        ],
    },
    {
        "title": "Free Particle",
        "V": free_V,
        "psi0": gaussian_wavepacket(x, -40.0, 3.5, 6.0),
        "markers": [],
    },
]

## 8. Shared Figure Setup

A helper that initialises a 2×2 matplotlib figure, plots the (scaled) potentials and markers, and returns the per-panel simulation state dictionaries. Reused by both animations below.

In [6]:
def build_figure_and_states():
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
    axes = axes.flatten()

    sim_states = []
    for ax, config in zip(axes, sim_configs):
        V_vals = config["V"]
        psi0   = config["psi0"]
        solver = build_thomas_solver(V_vals)

        max_V = np.max(V_vals)
        if max_V > 0:
            ax.plot(x, V_vals * (0.12 / max_V), color="#777777", linestyle="--")

        for marker in config["markers"]:
            if marker["type"] == "span":
                ax.axvspan(marker["x0"], marker["x1"], color=marker["color"], alpha=0.35)
            elif marker["type"] == "vline":
                ax.axvline(marker["x"], color=marker["color"], linestyle=":", alpha=0.9)

        line, = ax.plot(x, np.abs(psi0) ** 2, color="#1f77b4")
        ax.set_title(config["title"])
        ax.set_xlim(-L / 2, L / 2)
        ax.set_ylim(0, 0.45)
        ax.grid(alpha=0.2)

        sim_states.append({
            "psi":      psi0.copy(),
            "psi_init": psi0.copy(),
            "solver":   solver,
            "line":     line,
        })

    fig.tight_layout()
    return fig, axes, sim_states

## 9. Animation 1 — Forward Evolution

Each frame advances every panel by `steps_per_frame` Crank–Nicolson timesteps. The animation loops, resetting $\psi$ to $\psi_0$ at frame 0.

In [7]:
T0_forward = 1.0
time_steps_forward = int(T0_forward / dt)

fig1, axes1, states1 = build_figure_and_states()

def update_forward(frame):
    if frame == 0:
        for state in states1:
            state["psi"] = state["psi_init"].copy()
    for state in states1:
        for _ in range(steps_per_frame):
            state["psi"] = state["solver"](state["psi"])
        state["line"].set_ydata(np.abs(state["psi"]) ** 2)
    return [s["line"] for s in states1]

ani1 = animation.FuncAnimation(
    fig1, update_forward,
    frames=time_steps_forward,
    interval=10, blit=True, repeat=True
)

writer = FFMpegWriter(fps=60)
ani1.save("simulation_cycle.mp4", writer=writer, dpi=150)
plt.close(fig1)
print("Saved simulation_cycle.mp4")

Saved simulation_cycle.mp4


## 10. Time-Reversal Symmetry

The Schrödinger equation is **time-reversal invariant**. To see why, take the complex conjugate of the TDSE:

$$-i\hbar \frac{\partial \psi^*}{\partial t} = \hat{H}\psi^*$$

Substituting $t \to -t$ shows that $\psi^*(x, -t)$ satisfies the same equation as $\psi(x, t)$. Therefore, if we **conjugate** the wavefunction at some time $t_1$:

$$\psi \xrightarrow{\text{time-reverse}} \psi^*$$

the subsequent Crank–Nicolson evolution propagates the system *backwards* in time, exactly retracing the trajectory. The probability density $|\psi^*|^2 = |\psi|^2$ is unchanged at the moment of reversal, but the phase (and hence the momentum direction) flips: $e^{ik_0 x} \to e^{-ik_0 x}$.

In the animation below, the wavepacket evolves forward until `frame == 320`, at which point $\psi \to \psi^*$ is applied. The system then retraces its path back toward the initial state.

In [8]:
T0_reversible = 1.7
time_steps_reversible = int(T0_reversible / dt)

fig2, axes2, states2 = build_figure_and_states()

def update_reversible(frame):
    if frame == 0:
        for ax in axes2:
            for txt in ax.texts:
                txt.remove()
        for state in states2:
            state["psi"] = state["psi_init"].copy()

    if frame == 320:
        for ax in axes2:
            ax.text(0.5, 0.5, "Reversing Time",
                    transform=ax.transAxes, fontsize=16,
                    color="#ff5555", ha="center", va="center", alpha=0.8)
        for state in states2:
            state["psi"] = np.conj(state["psi"])

    for state in states2:
        for _ in range(steps_per_frame):
            state["psi"] = state["solver"](state["psi"])
        state["line"].set_ydata(np.abs(state["psi"]) ** 2)

    return [s["line"] for s in states2]

ani2 = animation.FuncAnimation(
    fig2, update_reversible,
    frames=time_steps_reversible,
    interval=10, blit=False, repeat=True
)

writer = FFMpegWriter(fps=60)
ani2.save("reversible.mp4", writer=writer, dpi=150)
plt.close(fig2)
print("Saved reversible.mp4")

Saved reversible.mp4
